In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("./data/output_from_KNIME/chem-space.csv")
print(df.shape)
df.head()

In [ ]:
'''
df= (
    df.groupby("type", group_keys=False)
      .apply(lambda x: x.sample(frac=0.1, random_state=42))
)
'''
df.shape


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 計算関数 ---
def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return [np.nan]*8
    return [
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        CalcTPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.FractionCSP3(mol),
        QED.qed(mol)
    ]

# --- 変数 ---
descriptor_names = ['MolecularWeight','LogP','HBD','HBA','TPSA','RotatableBonds','Fsp3','QED']
cont_vars = ['MolecularWeight','LogP','TPSA','Fsp3','QED']  # QEDを追加
disc_vars = ['HBD','HBA','RotatableBonds']
groups_violin = ['hit','FDA-approved','R-BIND2']  # バイオリンプロット用
groups_hist = ['R-BIND2','FDA-approved','hit']    # ヒストグラム用（hitが最後＝上に表示）
palette = {'hit':'turquoise','FDA-approved':'gray','R-BIND2':'orange'}

# --- データ処理 ---
df['type_for_plot'] = df['type'].replace({'internal': 'hit'})
df[descriptor_names] = df['SMILES'].apply(calc_descriptors).tolist()

# --- 外れ値ビニング ---
df['HBD'] = df['HBD'].apply(lambda x: '≥10' if x >= 10 else str(int(x)))
df['HBA'] = df['HBA'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))
df['RotatableBonds'] = df['RotatableBonds'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))

# --- 連続変数: violin + box (QEDも含めて統一表示) ---
# 5つのプロットなので、各プロットが正方形に近くなるように調整
fig, axes = plt.subplots(1, len(cont_vars), figsize=(25, 5))
for ax, var in zip(axes, cont_vars):
    sns.violinplot(
        x='type_for_plot', y=var,
        data=df[df['type_for_plot'].isin(groups_violin)],
        palette=palette, inner=None, ax=ax, alpha=0.5,
        order=groups_violin
    )
    sns.boxplot(
        x='type_for_plot', y=var,
        data=df[df['type_for_plot'].isin(groups_violin)],
        width=0.1, showcaps=True,
        boxprops={'facecolor':'none','edgecolor':'black','linewidth':4},
        whiskerprops={'linewidth':4,'color':'black'},
        capprops={'linewidth':4,'color':'black'},
        medianprops={'linewidth':4,'color':'black'},
        ax=ax,
        order=groups_violin
    )
    ax.set_title(var, fontsize=20, pad=20)
    ax.set_xlabel('')
    ax.set_ylabel(var, fontsize=20, labelpad=10)
    ax.tick_params(axis='x', labelsize=18, pad=8)
    ax.tick_params(axis='y', labelsize=18, pad=8)
plt.tight_layout()
plt.show()

# --- 離散変数: histogram ---
fig, axes = plt.subplots(1, len(disc_vars), figsize=(15, 5))
for ax, var in zip(axes, disc_vars):
    unique_vals = sorted(df[var].unique(), key=lambda x: (int(x.replace('≥','').strip('+')) if x.startswith('≥') else int(x)))
    width = 0.3
    offset = 0.2
    for i, grp in enumerate(groups_hist):  # R-BIND2, FDA, hitの順で描画（hitが最後＝上）
        subset = df[df['type_for_plot']==grp]
        counts = subset[var].value_counts(normalize=True).reindex(unique_vals).fillna(0)
        positions = np.arange(len(unique_vals)) + (i - 0.5)*offset
        ax.bar(
            positions,
            counts.values,
            width=width,
            alpha=0.7,
            color=palette[grp],
            edgecolor='black',
            linewidth=1,
            label=grp
        )
    ax.set_title(var, fontsize=20, pad=20)
    ax.set_xlabel(var, fontsize=20, labelpad=10)
    ax.set_ylabel('Density', fontsize=20, labelpad=10)
    ax.set_xticks(np.arange(len(unique_vals)))
    ax.set_xticklabels(unique_vals, fontsize=18)
    ax.tick_params(axis='x', labelsize=18, pad=8)
    ax.tick_params(axis='y', labelsize=18, pad=8)
plt.tight_layout()
plt.show()

df.shape

有意差検定つき

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

# --- 計算関数 ---
def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return [np.nan]*8
    return [
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        CalcTPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.FractionCSP3(mol),
        QED.qed(mol)
    ]

# --- 変数 ---
descriptor_names = ['MolecularWeight','LogP','HBD','HBA','TPSA','RotatableBonds','Fsp3','QED']
cont_vars = ['MolecularWeight','LogP','TPSA','Fsp3','QED']
disc_vars = ['HBD','HBA','RotatableBonds']
groups = ['hit','FDA-approved','R-BIND2']
palette = {'hit':'turquoise','FDA-approved':'gray','R-BIND2':'orange'}

# --- データ処理 ---
df['type_for_plot'] = df['type'].replace({'internal': 'hit'})
df[descriptor_names] = df['SMILES'].apply(calc_descriptors).tolist()

# --- 外れ値ビニング（離散変数） ---
df['HBD'] = df['HBD'].apply(lambda x: '≥10' if x >= 10 else str(int(x)))
df['HBA'] = df['HBA'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))
df['RotatableBonds'] = df['RotatableBonds'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))

# --- 有意差検定付き連続変数プロット ---
# 5つのプロットなので、各プロットが正方形に近くなるように調整
fig, axes = plt.subplots(1, len(cont_vars), figsize=(25, 5))
comparisons = [('hit', 'FDA-approved'), ('hit', 'R-BIND2')]

for ax, var in zip(axes, cont_vars):
    plot_data = df[df['type_for_plot'].isin(groups)][['type_for_plot', var]].dropna()

    # 描画
    sns.violinplot(
        x='type_for_plot', y=var,
        data=plot_data,
        palette=palette, inner=None, ax=ax, alpha=0.5,
        order=groups
    )
    sns.boxplot(
        x='type_for_plot', y=var,
        data=plot_data,
        width=0.1, showcaps=True,
        boxprops={'facecolor':'none','edgecolor':'black','linewidth':4},
        whiskerprops={'linewidth':4,'color':'black'},
        capprops={'linewidth':4,'color':'black'},
        medianprops={'linewidth':4,'color':'black'},
        ax=ax,
        order=groups
    )

    # 有意差アノテーション
    y_max = plot_data[var].max()
    y_min = plot_data[var].min()
    y_range = y_max - y_min
    y_level = y_max + 0.05 * y_range
    for i, (grp1, grp2) in enumerate(comparisons):
        x1, x2 = groups.index(grp1), groups.index(grp2)
        values1 = plot_data[plot_data['type_for_plot'] == grp1][var]
        values2 = plot_data[plot_data['type_for_plot'] == grp2][var]
        pval = mannwhitneyu(values1, values2, alternative='two-sided').pvalue
        # p値をアスタリスクで表示
        if pval > 0.05:
            p_text = "n.s."
        elif pval <= 0.001:
            p_text = "***"
        elif pval <= 0.01:
            p_text = "**"
        else:
            p_text = "*"
        # 線とテキスト描画
        ax.plot([x1, x1, x2, x2], [y_level, y_level+0.01*y_range, y_level+0.01*y_range, y_level], lw=1.5, color='black')
        ax.text((x1 + x2) / 2, y_level + 0.015 * y_range, p_text, ha='center', va='bottom', fontsize=16)
        y_level += 0.07 * y_range

    # 軸設定
    ax.set_title(var, fontsize=20, pad=20)
    ax.set_xlabel('')
    ax.set_ylabel(var, fontsize=20, labelpad=10)
    ax.tick_params(axis='x', labelsize=18, pad=8)
    ax.tick_params(axis='y', labelsize=18, pad=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- 離散変数の統計検定付きプロット（元のグラフ形式を保持） ---

# 統計検定用に元のデータ（ビニング前）を準備
df_original = df.copy()
df_original[descriptor_names] = df_original['SMILES'].apply(calc_descriptors).tolist()

# 統計検定結果を保存するリスト
stat_results = []

# グループ順序の設定
groups = ['hit','FDA-approved','R-BIND2']
groups_hist = ['R-BIND2','FDA-approved','hit']  # ヒストグラム用（hitが最後＝上に表示）

# 統計検定の実行
comparisons = [('hit', 'FDA-approved'), ('hit', 'R-BIND2')]
for var in disc_vars:
    for grp1, grp2 in comparisons:
        # 元の数値データを使用（ビニング前）
        values1 = df_original[df_original['type_for_plot'] == grp1][var].dropna()
        values2 = df_original[df_original['type_for_plot'] == grp2][var].dropna()
        
        if len(values1) > 0 and len(values2) > 0:
            stat, pval = mannwhitneyu(values1, values2, alternative='two-sided')
            
            # 結果を保存
            stat_results.append({
                'variable': var,
                'comparison': f'{grp1} vs {grp2}',
                'statistic': stat,
                'p_value': pval,
                'n1': len(values1),
                'n2': len(values2)
            })

# --- 元のヒストグラムプロット（hitが最後に描画される＝上に表示） ---
fig, axes = plt.subplots(1, len(disc_vars), figsize=(15, 5))
for ax, var in zip(axes, disc_vars):
    unique_vals = sorted(df[var].unique(), key=lambda x: (int(x.replace('≥','').strip('+')) if x.startswith('≥') else int(x)))
    width = 0.3
    offset = 0.2
    for i, grp in enumerate(groups_hist):  # R-BIND2, FDA, hitの順で描画（hitが最後＝上）
        subset = df[df['type_for_plot']==grp]
        counts = subset[var].value_counts(normalize=True).reindex(unique_vals).fillna(0)
        positions = np.arange(len(unique_vals)) + (i - 0.5)*offset
        ax.bar(
            positions,
            counts.values,
            width=width,
            alpha=0.7,
            color=palette[grp],
            edgecolor='black',
            linewidth=1
        )
    ax.set_title(var, fontsize=20, pad=20)
    ax.set_xlabel(var, fontsize=20, labelpad=10)
    ax.set_ylabel('Density', fontsize=20, labelpad=10)
    ax.set_xticks(np.arange(len(unique_vals)))
    ax.set_xticklabels(unique_vals, fontsize=18)
    ax.tick_params(axis='x', labelsize=18, pad=8)
    ax.tick_params(axis='y', labelsize=18, pad=8)

plt.tight_layout()
plt.show()

# --- 統計検定結果の表示（グラフの下） ---
print("=== Mann-Whitney U Test Results for Discrete Variables ===")
if stat_results:
    for result in stat_results:
        print(f"\\n{result['variable']} - {result['comparison']}:")
        print(f"  Sample sizes: n1={result['n1']}, n2={result['n2']}")
        print(f"  U-statistic: {result['statistic']:.2f}")
        print(f"  P-value: {result['p_value']:.4f}")
        if result['p_value'] <= 0.001:
            significance = "*** (p ≤ 0.001)"
        elif result['p_value'] <= 0.01:
            significance = "** (p ≤ 0.01)"
        elif result['p_value'] <= 0.05:
            significance = "* (p ≤ 0.05)"
        else:
            significance = "n.s. (p > 0.05)"
        print(f"  Significance: {significance}")
else:
    print("No statistical test results available.")

print(f"\\nDataset shape: {df.shape}")

In [ ]:
# Test the implementation to verify it works correctly
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu


# --- 計算関数 ---
def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return [np.nan]*8
    return [
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        CalcTPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.FractionCSP3(mol),
        QED.qed(mol)
    ]

# --- 変数 ---
descriptor_names = ['MolecularWeight','LogP','HBD','HBA','TPSA','RotatableBonds','Fsp3','QED']
disc_vars = ['HBD','HBA','RotatableBonds']
groups = ['hit','FDA-approved','R-BIND2']

# --- データ処理 ---
df['type_for_plot'] = df['type'].replace({'internal': 'hit'})
df[descriptor_names] = df['SMILES'].apply(calc_descriptors).tolist()

print("Data preparation completed")
print(f"Dataset shape: {df.shape}")
print(f"Groups: {df['type_for_plot'].value_counts()}")

# Test statistical test on a small sample
df_original = df.copy()
test_var = 'HBD'
grp1_data = df_original[df_original['type_for_plot'] == 'hit'][test_var].dropna()
grp2_data = df_original[df_original['type_for_plot'] == 'FDA-approved'][test_var].dropna()

print(f"\nTesting Mann-Whitney U test for {test_var}:")
print(f"Hit group size: {len(grp1_data)}")
print(f"FDA-approved group size: {len(grp2_data)}")

if len(grp1_data) > 0 and len(grp2_data) > 0:
    stat, pval = mannwhitneyu(grp1_data, grp2_data, alternative='two-sided')
    print(f"U-statistic: {stat:.2f}")
    print(f"P-value: {pval:.4f}")
    print("Statistical test implementation works correctly!")
else:
    print("Error: One or both groups have no data")

In [ ]:
# Define effect size calculation functions
import numpy as np
from scipy.stats import mannwhitneyu

def cliffs_delta(x, y):
    """
    Calculate Cliff's delta effect size.
    
    Cliff's delta is a non-parametric effect size measure that quantifies
    the amount of difference between two groups. It ranges from -1 to 1:
    - δ = 1: all values in x are larger than all values in y
    - δ = 0: no difference between groups
    - δ = -1: all values in x are smaller than all values in y
    
    Interpretation thresholds (Romano et al., 2006):
    - |δ| < 0.147: negligible
    - 0.147 ≤ |δ| < 0.33: small
    - 0.33 ≤ |δ| < 0.474: medium
    - |δ| ≥ 0.474: large
    
    Parameters
    ----------
    x : array-like
        Values from first group
    y : array-like
        Values from second group
    
    Returns
    -------
    float
        Cliff's delta value
    """
    nx = len(x)
    ny = len(y)
    
    # Count dominances: how many times x > y, and how many times x < y
    dominance = 0
    for xi in x:
        for yi in y:
            if xi > yi:
                dominance += 1
            elif xi < yi:
                dominance -= 1
    
    # Normalize by total number of comparisons
    delta = dominance / (nx * ny)
    return delta

def interpret_cliffs_delta(delta):
    """
    Interpret the magnitude of Cliff's delta effect size.
    
    Based on Romano et al. (2006) thresholds.
    
    Parameters
    ----------
    delta : float
        Cliff's delta value
    
    Returns
    -------
    str
        Interpretation category: 'negligible', 'small', 'medium', or 'large'
    """
    abs_delta = abs(delta)
    if abs_delta < 0.147:
        return "negligible"
    elif abs_delta < 0.33:
        return "small"
    elif abs_delta < 0.474:
        return "medium"
    else:
        return "large"

# Test the functions
print("Effect size functions defined successfully!")
print("\nExample test:")
test_x = np.array([1, 2, 3, 4, 5])
test_y = np.array([3, 4, 5, 6, 7])
test_delta = cliffs_delta(test_x, test_y)
print(f"Cliff's delta = {test_delta:.3f} ({interpret_cliffs_delta(test_delta)} effect)")

In [ ]:
# Install openpyxl if not available
import sys
try:
    import openpyxl
    print("openpyxl is already installed")
except ImportError:
    print("Installing openpyxl...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    print("openpyxl installed successfully!")

In [ ]:
# Export to Excel with NO missing values (fixed column structure)
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

all_vars = ['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']
comparisons = [('hit', 'FDA-approved'), ('hit', 'R-BIND2')]

comprehensive_results = []

for var in all_vars:
    for grp1, grp2 in comparisons:
        data1 = df_stats[df_stats['type_for_plot'] == grp1][var].dropna()
        data2 = df_stats[df_stats['type_for_plot'] == grp2][var].dropna()
        
        if len(data1) == 0 or len(data2) == 0:
            continue
        
        n1, n2 = len(data1), len(data2)
        median1, median2 = data1.median(), data2.median()
        q1_1, q3_1 = data1.quantile(0.25), data1.quantile(0.75)
        q1_2, q3_2 = data2.quantile(0.25), data2.quantile(0.75)
        iqr1, iqr2 = q3_1 - q1_1, q3_2 - q1_2
        mean1, std1 = data1.mean(), data1.std()
        mean2, std2 = data2.mean(), data2.std()
        
        stat, pval = mannwhitneyu(data1, data2, alternative='two-sided')
        delta = cliffs_delta(data1.values, data2.values)
        delta_interp = interpret_cliffs_delta(delta)
        
        sig = "***" if pval <= 0.001 else "**" if pval <= 0.01 else "*" if pval <= 0.05 else "n.s."
        
        # Consistent naming: Group1/Group2 (NO missing values!)
        comprehensive_results.append({
            'Variable': var,
            'Comparison': f'{grp1} vs {grp2}',
            'Group1': grp1,
            'Group1_n': n1,
            'Group1_Median': round(median1, 3),
            'Group1_Q1': round(q1_1, 3),
            'Group1_Q3': round(q3_1, 3),
            'Group1_IQR': round(iqr1, 3),
            'Group1_Mean': round(mean1, 3),
            'Group1_SD': round(std1, 3),
            'Group2': grp2,
            'Group2_n': n2,
            'Group2_Median': round(median2, 3),
            'Group2_Q1': round(q1_2, 3),
            'Group2_Q3': round(q3_2, 3),
            'Group2_IQR': round(iqr2, 3),
            'Group2_Mean': round(mean2, 3),
            'Group2_SD': round(std2, 3),
            'U_statistic': round(stat, 2),
            'p_value': pval,
            'p_formatted': f"{pval:.4e}" if pval < 0.0001 else f"{pval:.4f}",
            'Significance': sig,
            'Cliffs_delta': round(delta, 3),
            'Effect_size': delta_interp,
            'Biological_relevance': 'High' if abs(delta) >= 0.33 else 'Moderate' if abs(delta) >= 0.147 else 'Low'
        })

results_df = pd.DataFrame(comprehensive_results)
output_excel = "./data/output_from_code/S2_Statistical_Analysis_Complete_pre.xlsx"

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Complete_Statistics', index=False)
    
    summary_df = results_df[['Variable', 'Comparison', 'Group1', 'Group1_n', 'Group1_Median', 'Group1_IQR',
                            'Group2', 'Group2_n', 'Group2_Median', 'Group2_IQR',
                            'p_formatted', 'Significance', 'Cliffs_delta', 'Effect_size', 'Biological_relevance']]
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

wb = load_workbook(output_excel)
for sheet_name in ['Complete_Statistics', 'Summary']:
    ws = wb[sheet_name]
    fill = PatternFill(start_color="366092" if sheet_name == 'Complete_Statistics' else "70AD47", 
                      end_color="366092" if sheet_name == 'Complete_Statistics' else "70AD47", 
                      fill_type="solid")
    for cell in ws[1]:
        cell.fill = fill
        cell.font = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    for column in ws.columns:
        max_len = max(len(str(cell.value)) for cell in column)
        ws.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)

wb.save(output_excel)

print("="*80)
print("EXCEL OUTPUT FIXED - NO MISSING VALUES!")
print("="*80)
print(f"File: {output_excel}")
print(f"Comparisons: {len(results_df)}")
print("\nKey fix: Group1/Group2 naming = ALL cells have values!")
print("Sheets: Complete_Statistics, Summary")